In [1]:
import pandas as pd

df = pd.read_parquet("datasets/yellow_tripdata_2026-01.parquet")

df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')

In [26]:
# create a new column trip_duration_min by calculating the difference between tpep_dropoff_datetime and tpep_pickup_datetime in minutes.

df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# build a valid dataframe with only the trip_duration, trip_distance, and fare_amount, total_amount, tpep_dropoff_datetime, tpep_pickup_datetime columns.
valid_mask = (
    (df["trip_duration_min"] > 0)
    & (df["trip_distance"] > 0)
    & (df["fare_amount"] > 0)
    & (df["total_amount"] > 0)
    & (df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"])
)

df_valid = df[valid_mask].copy()
df_invalid = df[~valid_mask].copy()

print(valid_mask)

0           True
1           True
2           True
3           True
4           True
           ...  
3724884     True
3724885    False
3724886     True
3724887     True
3724888     True
Length: 3724889, dtype: bool


In [12]:
df_valid.groupby("PULocationID").size().sort_values(ascending=False).head(10)

PULocationID
237    152521
236    145698
132    141596
161    139203
186    105633
142    104666
162    103673
230    100828
79      96382
239     92410
dtype: int64

In [11]:
df_valid.groupby(["PULocationID", "DOLocationID"]).size().sort_values(
    ascending=False
).head(10)

PULocationID  DOLocationID
237           236             23272
236           237             20164
              236             16004
237           237             14806
161           237              9969
237           161              9155
142           239              8734
239           238              8522
161           236              8250
141           236              8216
dtype: int64

In [28]:
print("Original rows:", len(df))
print("Valid rows:", len(df_valid))
print("Invalid rows:", len(df_invalid))
print("Valid %:", len(df_valid) / len(df) * 100)

print(
    df_valid[
        ["trip_distance", "fare_amount", "total_amount", "trip_duration_min"]
    ].describe()
)

Original rows: 3724889
Valid rows: 3516922
Invalid rows: 207967
Valid %: 94.41682691752695
       trip_distance   fare_amount  total_amount  trip_duration_min
count   3.516922e+06  3.516922e+06  3.516922e+06       3.516922e+06
mean    6.757459e+00  2.109072e+01  2.967819e+01       1.746949e+01
std     6.677936e+02  1.753461e+01  2.105659e+01       2.429999e+01
min     1.000000e-02  1.000000e-02  7.000000e-01       1.666667e-02
25%     1.090000e+00  1.000000e+01  1.722000e+01       8.333333e+00
50%     1.900000e+00  1.560000e+01  2.310000e+01       1.351667e+01
75%     3.880000e+00  2.610000e+01  3.376000e+01       2.133333e+01
max     2.690975e+05  2.555200e+03  2.560200e+03       5.293333e+03


In [43]:
df_valid["tpep_pickup_datetime"] = pd.to_datetime(df_valid["tpep_pickup_datetime"])
# print(df_valid["tpep_pickup_datetime"].min())
df_valid = df_valid.sort_values(by="tpep_pickup_datetime", ascending=True)

df_valid.head()

df_invalid["tpep_pickup_datetime"] = pd.to_datetime(df_invalid["tpep_pickup_datetime"])
# print(df_invalid["tpep_pickup_datetime"].min())
df_invalid = df_invalid.sort_values(by="tpep_pickup_datetime", ascending=True)


df_valid.to_parquet("datasets/valid_trips.parquet", index=False)
df_invalid.to_parquet("datasets/invalid_trips.parquet", index=False)

In [39]:
df_invalid.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min
980,2,2025-12-31 23:57:29,2025-12-31 23:57:32,2.0,0.01,2.0,N,132,264,3,...,0.0,-0.5,0.0,0.0,-1.0,-71.50,0.0,0.0,0.00,0.050000
2640643,2,2026-01-01 00:00:56,2026-01-01 00:17:26,NaN,0.00,NaN,NaN,42,212,0,...,0.0,0.5,0.0,0.0,1.0,21.50,NaN,NaN,0.00,16.500000
4288,2,2026-01-01 00:01:11,2026-01-01 00:01:17,1.0,0.00,5.0,N,143,143,1,...,0.0,0.0,0.0,0.0,1.0,36.95,2.5,0.0,0.00,0.100000
5065,2,2026-01-01 00:01:19,2026-01-01 00:04:08,1.0,0.05,1.0,N,48,163,2,...,-1.0,-0.5,0.0,0.0,-1.0,-10.15,-2.5,0.0,-0.75,2.816667
428,2,2026-01-01 00:01:24,2026-01-01 00:02:02,1.0,0.03,1.0,N,162,162,4,...,-1.0,-0.5,0.0,0.0,-1.0,-8.75,-2.5,0.0,-0.75,0.633333
